# kbprojection experiment runner

This notebook runs the experiment pipeline:

1. Load an NLI dataset (`SICKLoader` or `SNLILoader`).
2. Run a no-KB LangPro baseline.
3. Generate KB injections with an LLM.
4. Optionally test raw LLM KB.
5. Filter/post-process KB injections.
6. Re-run LangPro with filtered KB.
7. Save, summarize, and inspect results.

The notebook defaults to `RUN_EXPERIMENT = False` so it will not make live LangPro/LLM calls until you explicitly opt in.

Use a Python 3.10+ kernel. This repository uses modern type hints such as `str | dict` that do not import under Python 3.9.


## 1. Imports and environment check


In [1]:
import csv
import importlib.util
import json
import os
import sys
from collections import Counter
from datetime import datetime
from pathlib import Path
from pprint import pprint

if sys.version_info < (3, 10):
    raise RuntimeError("Use a Python 3.10+ kernel for this repository.")

# Runtime/path setup. In Colab, set KBPROJECTION_RUNTIME=colab and
# KBPROJECTION_PROJECT_ROOT=/content/drive/MyDrive/kbprojection before this cell,
# or let configure_runtime auto-detect Colab and mount Drive.
_candidate_roots = [
    Path(os.environ.get("KBPROJECTION_PROJECT_ROOT", Path.cwd())).expanduser().resolve(),
    Path.cwd().resolve(),
    Path("/content/drive/MyDrive/kbprojection"),
]
for _candidate_root in _candidate_roots:
    if _candidate_root.exists() and str(_candidate_root) not in sys.path:
        sys.path.insert(0, str(_candidate_root))

import kbprojection
from kbprojection import SICKLoader, SNLILoader
from kbprojection.runtime import configure_runtime
from kbprojection.models import (
    ExperimentResult,
    ExperimentStatus,
    ProblemConfig,
    TestMode,
)
from kbprojection.orchestration import (
    collect_kb_helpful_examples_random,
    process_kb_examples,
)

RUNTIME_PATHS = configure_runtime()
PROJECT_ROOT = RUNTIME_PATHS.project_root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("kbprojection version:", kbprojection.__version__)
print("Runtime:", RUNTIME_PATHS.runtime)
print("Project root:", PROJECT_ROOT)
print("Working directory:", Path.cwd())


Python: 3.11.13
kbprojection: /Users/jorrytdejong/Documents/RNL paper publication/kbprojection/__init__.py
Working directory: /Users/jorrytdejong/Documents/RNL paper publication


## 2. Experiment parameters

Edit this cell before running the experiment. The safest first run is a small SICK dev run with low `MAX_CHECKED`.


In [10]:
# Safety switch: set True only when you are ready for live LangPro + LLM calls.
RUN_EXPERIMENT = True

# Optional local checks that may download NLTK tokenizers/corpora the first time.
RUN_LOCAL_PREFLIGHT = False

# Dataset settings.
DATASET_NAME = "sick"       # "sick" or "snli"
SPLIT = "dev"               # SICK: train/dev/test; SNLI: train/dev/test
DATA_DIR = RUNTIME_PATHS.data_dir
LABEL_FILTER = {"entailment", "contradiction"}

# If provided, these exact IDs are processed instead of random sampling.
# Example SICK ids are simple strings like "1", "2", ... after the dataset is downloaded.
PROBLEM_IDS = []

# Sampling/result limits.
YIELD_MODE = "helpful"      # "helpful" yields only fixed examples; "all" yields every processed problem
MAX_RESULTS = 5              # helpful matches if YIELD_MODE="helpful"; processed results if YIELD_MODE="all"
MAX_CHECKED = 25             # upper bound on problems attempted

# LLM settings. GenericAIClient accepts: "openai", "openrouter", "gemini", or "claude".
LLM_PROVIDER = "openai"
MODEL = "gpt-4o"
PROMPT_STYLE = "icl"        # usually "icl" or "cot"; legacy_* prompts also exist

# Pipeline settings.
TEST_MODE = "both"          # "no_kb", "raw_kb", "filtered", "both", or "full"
RUN_ABLATION = False         # True is slower; only useful after KB fixes a problem
VERBOSE = True

# LangPro endpoint. The notebook patches orchestration calls to use this value.
LANGPRO_ENDPOINT = "https://langpro-annotator.hum.uu.nl/langpro-api/prove/"

# Output locations.
CACHE_DIR = RUNTIME_PATHS.cache_root
RESULTS_DIR = RUNTIME_PATHS.results_dir
RUN_NAME = f"{DATASET_NAME}_{SPLIT}_{YIELD_MODE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print("Configured run:", RUN_NAME)
print("RUN_EXPERIMENT:", RUN_EXPERIMENT)


Configured run: sick_dev_helpful_20260427_123621
RUN_EXPERIMENT: True


## 3. Preflight checks

This checks importability, provider keys, and output directories. It does not call LangPro or an LLM.


In [11]:
def has_package(name):
    try:
        return importlib.util.find_spec(name) is not None
    except ModuleNotFoundError:
        return False

provider_env = {
    "openai": "OPENAI_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
    "gemini": "GEMINI_API_KEY",
    "claude": "ANTHROPIC_API_KEY",
}

print("Package checks:")
for package in ["requests", "pydantic", "nltk", "openai", "anthropic", "google.genai"]:
    print(f"  {package:14s}", "OK" if has_package(package) else "missing")

key_name = provider_env.get(LLM_PROVIDER)
print("\nProvider:", LLM_PROVIDER)
if key_name:
    print(f"API key {key_name}:", "set" if os.environ.get(key_name) else "not set")
else:
    print("Unknown provider. Expected one of:", sorted(provider_env))

CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("\nCache directory:", CACHE_DIR.resolve())
print("Results directory:", RESULTS_DIR.resolve())


Package checks:
  requests       OK
  pydantic       OK
  nltk           OK
  openai         OK
  anthropic      OK
  google.genai   OK

Provider: openai
API key OPENAI_API_KEY: set

Cache directory: /Users/jorrytdejong/Documents/RNL paper publication/experiment_cache
Results directory: /Users/jorrytdejong/Documents/RNL paper publication/experiment_results


## 4. Build dataset loader and experiment config

This constructs the loader/config objects. Dataset files are downloaded later only when the loader is actually used.


In [12]:
def build_dataset(name: str):
    name = name.lower().strip()
    if name == "sick":
        return SICKLoader(data_dir=DATA_DIR / "sick")
    if name == "snli":
        return SNLILoader(data_dir=DATA_DIR / "snli")
    raise ValueError("DATASET_NAME must be 'sick' or 'snli'.")


def enum_value(x):
    return x.value if hasattr(x, "value") else x


dataset = build_dataset(DATASET_NAME)
config = ProblemConfig(
    llm_provider=LLM_PROVIDER,
    model=MODEL,
    prompt_style=PROMPT_STYLE,
    test_mode=TestMode(TEST_MODE),
    run_ablation=RUN_ABLATION,
    verbose=VERBOSE,
)

print("Dataset loader:", dataset.__class__.__name__)
print("Dataset directory:", dataset.data_dir)
print("Splits:", dataset._get_splits())
print("ProblemConfig:")
pprint(config.model_dump())


Dataset loader: SICKLoader
Dataset directory: data/sick
Splits: ['train', 'dev', 'test']
ProblemConfig:
{'llm_provider': 'openai',
 'model': 'gpt-4o',
 'post_process': True,
 'prompt_style': 'icl',
 'run_ablation': False,
 'test_mode': <TestMode.BOTH: 'both'>,
 'verbose': True}


## 5. Patch LangPro endpoint used by orchestration

The repository's `process_single_problem` function uses a module-level `langpro_api_call`. This cell patches that call so the runner respects `LANGPRO_ENDPOINT`.


In [13]:
import kbprojection.orchestration as orchestration
from kbprojection.langpro import langpro_api_call as original_langpro_api_call


def langpro_api_call_with_runner_endpoint(premises, hypothesis, **kwargs):
    kwargs.setdefault("endpoint", LANGPRO_ENDPOINT)
    return original_langpro_api_call(premises, hypothesis, **kwargs)

orchestration.langpro_api_call = langpro_api_call_with_runner_endpoint
print("LangPro endpoint patched to:", LANGPRO_ENDPOINT)


LangPro endpoint patched to: https://langpro-annotator.hum.uu.nl/langpro-api/prove/


## 6. Optional local preflight without LangPro/LLM

This validates prompt filling, mock LLM output extraction, and KB filtering on a hand-written example. It may download NLTK resources the first time, so it is gated by `RUN_LOCAL_PREFLIGHT`.


In [14]:
if not RUN_LOCAL_PREFLIGHT:
    print("Local preflight skipped. Set RUN_LOCAL_PREFLIGHT = True to run it.")
else:
    from kbprojection.llm import extract_kb_from_output
    from kbprojection.models import NLILabel, NLIProblem
    from kbprojection.prompts import fill_prompt
    from kbprojection.filtering import pipeline_filter_kb_injections

    demo_problem = NLIProblem(
        id="local-demo",
        premises=["A little girl in pink boots runs down the street."],
        hypothesis="A human is running outdoors.",
        gold_label=NLILabel.ENTAILMENT,
        dataset="manual",
        split="demo",
    )

    prompt = fill_prompt(PROMPT_STYLE, demo_problem.premises, demo_problem.hypothesis)
    mock_output = """
    [KB_START]
    isa_wn(girl, human)
    isa_wn(street, outdoors)
    [KB_END]
    """
    raw_kb = extract_kb_from_output(mock_output)
    filtered = pipeline_filter_kb_injections(raw_kb, demo_problem.premises, demo_problem.hypothesis)

    print("Prompt preview:")
    print(prompt[-500:])
    print("\nRaw KB:", raw_kb)
    print("Filtered KB:", [str(item) for item in filtered])


Local preflight skipped. Set RUN_LOCAL_PREFLIGHT = True to run it.


## 7. Optional dataset preview

Set `PREVIEW_DATASET = True` to download/load the selected split and print a few examples before launching the experiment.


In [16]:
PREVIEW_DATASET = True
PREVIEW_N = 3

if not PREVIEW_DATASET:
    print("Dataset preview skipped. Set PREVIEW_DATASET = True to run it.")
else:
    dataset.load(splits=[SPLIT])
    print(f"First {PREVIEW_N} examples from {DATASET_NAME}/{SPLIT}:")
    for index, problem in enumerate(dataset.iter_problems(SPLIT, label_filter=LABEL_FILTER), start=1):
        print("-" * 80)
        print("id:", problem.id)
        print("gold:", problem.gold_label.value)
        print("premises:", problem.premises)
        print("hypothesis:", problem.hypothesis)
        if index >= PREVIEW_N:
            break


[SICKLoader] Downloading SICK_train.txt...
[Download] Downloading https://raw.githubusercontent.com/brmson/dataset-sts/master/data/sts/sick2014/SICK_train.txt to data/sick/SICK_train.txt...
[Download] Finished: data/sick/SICK_train.txt
[SICKLoader] Downloading SICK_trial.txt...
[Download] Downloading https://raw.githubusercontent.com/brmson/dataset-sts/master/data/sts/sick2014/SICK_trial.txt to data/sick/SICK_trial.txt...
[Download] Finished: data/sick/SICK_trial.txt
[SICKLoader] Downloading SICK_test_annotated.txt...
[Download] Downloading https://raw.githubusercontent.com/brmson/dataset-sts/master/data/sts/sick2014/SICK_test_annotated.txt to data/sick/SICK_test_annotated.txt...
[Download] Finished: data/sick/SICK_test_annotated.txt
[SICKLoader] Loaded 500 problems from dev.
First 3 examples from sick/dev:
--------------------------------------------------------------------------------
id: 4
gold: contradiction
premises: ['The young boys are playing outdoors and the man is smiling nea

## 8. Run the experiment

This is the live runner cell. With `RUN_EXPERIMENT = True`, it will call LangPro and your configured LLM provider. Results are kept in memory as `results` and each processed problem may also be cached in `CACHE_DIR`.


In [17]:
results = []
problem_ids = PROBLEM_IDS or None

if not RUN_EXPERIMENT:
    print("Experiment not launched. Set RUN_EXPERIMENT = True in the parameter cell, then rerun from there.")
else:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

    if YIELD_MODE == "helpful":
        generator = collect_kb_helpful_examples_random(
            dataset=dataset,
            config=config,
            split=SPLIT,
            label_filter=LABEL_FILTER,
            max_matches=MAX_RESULTS,
            max_checked=MAX_CHECKED,
            problem_ids=problem_ids,
            cache_dir=CACHE_DIR,
        )
    elif YIELD_MODE == "all":
        generator = process_kb_examples(
            dataset=dataset,
            config=config,
            split=SPLIT,
            label_filter=LABEL_FILTER,
            max_matches=MAX_RESULTS,
            max_checked=MAX_CHECKED,
            problem_ids=problem_ids,
            cache_dir=CACHE_DIR,
        )
    else:
        raise ValueError("YIELD_MODE must be 'helpful' or 'all'.")

    for item in generator:
        results.append(item)

    print(f"Collected {len(results)} result(s).")


[kb-processor] Processing split='dev', labels={'entailment', 'contradiction'}
[kb-processor] Config: mode=both, ablation=False
[kb-processor] Using random sampling.

[kb-processor] #1 | Key: 1690 | Gold: entailment

[process] Key: 1690 | Gold: entailment
[process] Premises: ['The milk is being drunk by a cat']
[process] Hypothesis: The cat is drinking some milk
[process] Mode: both | Ablation: False
  [no-KB] Calling LangPro...
  [no-KB] Predicted: -
  [no-KB] Call failed.
[cache] Result cached to experiment_cache/sick_dev_1690.json

[kb-processor] #2 | Key: 2001 | Gold: entailment

[process] Key: 2001 | Gold: entailment
[process] Premises: ['The horse is being ridden by a man']
[process] Hypothesis: A man is riding the horse
[process] Mode: both | Ablation: False
  [no-KB] Calling LangPro...
  [no-KB] Predicted: -
  [no-KB] Call failed.
[cache] Result cached to experiment_cache/sick_dev_2001.json

[kb-processor] #3 | Key: 1266 | Gold: entailment

[process] Key: 1266 | Gold: entailment

## 9. Optionally load cached results

If the live run was skipped or interrupted, this cell can load previously cached `ExperimentResult` JSON files from `CACHE_DIR`.


In [18]:
LOAD_CACHED_RESULTS_IF_EMPTY = True

if results:
    print(f"Already have {len(results)} in-memory result(s); not loading cache.")
elif not LOAD_CACHED_RESULTS_IF_EMPTY:
    print("Cache loading skipped.")
elif not CACHE_DIR.exists():
    print("No cache directory found:", CACHE_DIR)
else:
    cache_files = sorted(CACHE_DIR.glob("*.json"))
    for path in cache_files:
        try:
            results.append(ExperimentResult.model_validate_json(path.read_text(encoding="utf-8")))
        except Exception as exc:
            print("Could not load", path, "->", exc)
    print(f"Loaded {len(results)} cached result(s) from {CACHE_DIR}.")


Loaded 25 cached result(s) from experiment_cache.


## 10. Summarize results


In [19]:
def result_to_row(result):
    problem = result.problem
    return {
        "problem_id": problem.id,
        "dataset": problem.dataset,
        "split": problem.split,
        "gold_label": enum_value(problem.gold_label),
        "pred_no_kb": enum_value(result.pred_no_kb) if result.pred_no_kb else None,
        "pred_with_raw_kb": enum_value(result.pred_with_raw_kb) if result.pred_with_raw_kb else None,
        "pred_with_filtered_kb": enum_value(result.pred_with_kb) if result.pred_with_kb else None,
        "final_status": enum_value(result.final_status),
        "fixed_by": result.fixed_by,
        "kb_raw": " | ".join(result.kb_raw or []),
        "kb_filtered": " | ".join(result.kb_filtered or []),
        "premises": " || ".join(problem.premises),
        "hypothesis": problem.hypothesis,
    }

rows = [result_to_row(result) for result in results]
print("Result count:", len(rows))
print("Final-status counts:")
pprint(Counter(row["final_status"] for row in rows))
print("Fixed-by counts:")
pprint(Counter(row["fixed_by"] for row in rows if row["fixed_by"]))

if rows:
    try:
        import pandas as pd
        df = pd.DataFrame(rows)
        display(df)
    except Exception:
        print("\nFirst rows:")
        pprint(rows[:5])
else:
    print("No rows yet. Run the experiment or load cached results.")


Result count: 25
Final-status counts:
Counter({'error_no_kb': 25})
Fixed-by counts:
Counter()

First rows:
[{'dataset': 'sick',
  'final_status': 'error_no_kb',
  'fixed_by': None,
  'gold_label': 'contradiction',
  'hypothesis': 'There is no woman combing her hair',
  'kb_filtered': '',
  'kb_raw': '',
  'pred_no_kb': '-',
  'pred_with_filtered_kb': None,
  'pred_with_raw_kb': None,
  'premises': 'A woman is combing her hair',
  'problem_id': '1178',
  'split': 'dev'},
 {'dataset': 'sick',
  'final_status': 'error_no_kb',
  'fixed_by': None,
  'gold_label': 'contradiction',
  'hypothesis': 'A woman is dancing and singing with other women',
  'kb_filtered': '',
  'kb_raw': '',
  'pred_no_kb': '-',
  'pred_with_filtered_kb': None,
  'pred_with_raw_kb': None,
  'premises': 'A woman is dancing and singing alone',
  'problem_id': '1236',
  'split': 'dev'},
 {'dataset': 'sick',
  'final_status': 'error_no_kb',
  'fixed_by': None,
  'gold_label': 'entailment',
  'hypothesis': 'A band is play

## 11. Inspect fixed examples


In [20]:
fixed_results = [result for result in results if enum_value(result.final_status).startswith("fixed")]
print("Fixed examples:", len(fixed_results))

for result in fixed_results[:5]:
    problem = result.problem
    print("=" * 100)
    print("ID:", problem.id)
    print("Gold:", problem.gold_label.value)
    print("No-KB prediction:", enum_value(result.pred_no_kb) if result.pred_no_kb else None)
    print("Raw-KB prediction:", enum_value(result.pred_with_raw_kb) if result.pred_with_raw_kb else None)
    print("Filtered-KB prediction:", enum_value(result.pred_with_kb) if result.pred_with_kb else None)
    print("Fixed by:", result.fixed_by)
    print("Premises:", problem.premises)
    print("Hypothesis:", problem.hypothesis)
    print("Raw KB:", result.kb_raw)
    print("Filtered KB:", result.kb_filtered)
    if result.essential_kb:
        print("Essential KB:", result.essential_kb)


Fixed examples: 0


## 12. Export run artifacts

Writes a JSONL file containing full `ExperimentResult` objects and a CSV file containing a flat summary table.


In [21]:
if not results:
    print("No results to export.")
else:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    jsonl_path = RESULTS_DIR / f"{RUN_NAME}.jsonl"
    csv_path = RESULTS_DIR / f"{RUN_NAME}.csv"

    with jsonl_path.open("w", encoding="utf-8") as f:
        for result in results:
            f.write(result.model_dump_json() + "\n")

    rows = [result_to_row(result) for result in results]
    with csv_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    print("Wrote:", jsonl_path.resolve())
    print("Wrote:", csv_path.resolve())


Wrote: /Users/jorrytdejong/Documents/RNL paper publication/experiment_results/sick_dev_helpful_20260427_123621.jsonl
Wrote: /Users/jorrytdejong/Documents/RNL paper publication/experiment_results/sick_dev_helpful_20260427_123621.csv


## 13. Recommended next moves

After a first run, useful variations are:

- Change `YIELD_MODE` from `helpful` to `all` to measure all outcomes, not just fixed cases.
- Increase `MAX_CHECKED` for a broader sample.
- Toggle `TEST_MODE` between `raw_kb`, `filtered`, and `both` to isolate filtering effects.
- Set `RUN_ABLATION = True` once you have fixed examples and want minimal sufficient KB subsets.
- Pin `PROBLEM_IDS` to rerun interesting or failed cases exactly.
